# MachineGuard: Data Exploration

This notebook explores the AI4I 2020 Predictive Maintenance Dataset
before building the preprocessing and machine-learning pipeline.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
data_path = Path("../data/raw/ai4i2020.csv")

df = pd.read_csv(data_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 10,000
Columns: 14


In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 1. Data quality

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [6]:
data_quality = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(),
})

data_quality

,data_type,missing_values,unique_values
UDI,int64,0,10000
Product ID,str,0,10000
Type,str,0,3
Air temperature [K],float64,0,93
Process temperature [K],float64,0,82
Rotational speed [rpm],int64,0,941
Torque [Nm],float64,0,577
Tool wear [min],int64,0,246
Machine failure,int64,0,2
TWF,int64,0,2


In [5]:
duplicate_count = df.duplicated().sum()

print(f"Missing values: {df.isna().sum().sum()}")
print(f"Duplicate rows: {duplicate_count}")

Missing values: 0
Duplicate rows: 0


## 2. Target distribution

The prediction target is `Machine failure`. We inspect its distribution
because rare failures can make accuracy misleading.

In [7]:
target_summary = pd.DataFrame({
    "count": df["Machine failure"].value_counts(),
    "percentage": df["Machine failure"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2),
})

target_summary.index = ["No failure", "Failure"]
target_summary

,count,percentage
No failure,9661,96.61
Failure,339,3.39


## 3. Target leakage

The dataset contains five failure-mode indicators: `TWF`, `HDF`, `PWF`,
`OSF`, and `RNF`. These columns describe specific causes of machine
failure and therefore reveal information about the target.

They must not be used as model inputs because they would not be available
when predicting failure in advance.

In [8]:
failure_mode_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]

failure_mode_summary = pd.DataFrame({
    "failure_count": df[failure_mode_columns].sum()
}).sort_values("failure_count", ascending=False)

failure_mode_summary

,failure_count
HDF,115
OSF,98
PWF,95
TWF,46
RNF,19


In [9]:
any_failure_mode = df[failure_mode_columns].max(axis=1)

pd.crosstab(
    df["Machine failure"],
    any_failure_mode,
    rownames=["Machine failure"],
    colnames=["Any failure-mode flag"],
)

Any failure-mode flag,0,1
Machine failure,,
0,9643,18
1,9,330


### Feature selection

The following columns are excluded:

- `UDI`: row identifier with no predictive meaning
- `Product ID`: unique product identifier
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF`: failure-mode indicators that leak
  information about the prediction target

In [10]:
feature_columns = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

target_column = "Machine failure"

X = df[feature_columns].copy()
y = df[target_column].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

X.head()

Feature matrix shape: (10000, 6)
Target shape: (10000,)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


## 4. Exploration summary

- The dataset contains 10,000 observations and 14 original columns.
- No missing values or duplicate rows were found.
- Machine failure is strongly imbalanced: 339 failures (3.39%) and
  9,661 non-failures (96.61%).
- Accuracy alone is therefore not an appropriate primary evaluation metric.
- `UDI` and `Product ID` are excluded because they are identifiers.
- `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are excluded because they reveal
  failure-mode information and would cause target leakage.
- Six valid input features remain: product type, two temperatures,
  rotational speed, torque, and tool wear.
- Models will be evaluated primarily using precision, recall, F1 score,
  and precision-recall AUC.